<a href="https://colab.research.google.com/github/rileighdethy/mgmt467-analytics-portfolio/blob/main/Unit2_Lab1_PromptPlusExamples_Colab_Kaggle_GCS_BQ_DQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MGMT 467 — Prompt-Driven Lab (with Commented Examples)
## Kaggle ➜ Google Cloud Storage ➜ BigQuery ➜ Data Quality (DQ)

**How to use this notebook**
- Each section gives you a **Build Prompt** to paste into Gemini/Vertex AI (or Gemini in Colab).
- Below each prompt, you’ll see a **commented example** of what a good LLM answer might look like.
- **Do not** just uncomment and run. Use the prompt to generate your own code, then compare to the example.
- After every step, run the **Verification Prompt**, and write the **Reflection** in Markdown.

> Goal today: Download the Netflix dataset (Kaggle) → Stage on GCS → Load into BigQuery → Run DQ profiling (missingness, duplicates, outliers, anomaly flags).


### Academic integrity & LLM usage
- Use the prompts here to generate your own code cells.
- Read concept notes and write the reflection answers in your own words.
- Keep credentials out of code. Upload `kaggle.json` when asked.


## Learning objectives
1) Explain **why** we stage data in GCS and load it to BigQuery.  
2) Build an **idempotent**, auditable pipeline.  
3) Diagnose **missingness**, **duplicates**, and **outliers** and justify cleaning choices.  
4) Connect DQ decisions to **business/ML impact**.


## 0) Environment setup — What & Why
Authenticate Colab to Google Cloud so we can use `gcloud`, GCS, and BigQuery. Set **PROJECT_ID** and **REGION** once for consistency (cost/latency).

### Build Prompt (paste to LLM)
You are my cloud TA. Generate a single **Colab code cell** that:
1) Authenticates to Google Cloud in Colab,  
2) Prompts for `PROJECT_ID` via `input()` and sets `REGION="us-central1"` (editable),  
3) Exports `GOOGLE_CLOUD_PROJECT`,  
4) Runs `gcloud config set project $GOOGLE_CLOUD_PROJECT`,  
5) Prints both values. Add 2–3 comments explaining what/why.
End with a comment: `# Done: Auth + Project/Region set`.


In [2]:
# Authenticate to Google Cloud
# This will prompt you to log in and select your GCP project.
from google.colab import auth
auth.authenticate_user()

import os
# Prompt for the Project ID and set the Region
PROJECT_ID = input("Enter your GCP Project ID: ").strip()
REGION = "us-central1"  # You can change this region if needed

# Export the Project ID and Region as environment variables for use in shell commands
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["REGION"] = REGION # Export REGION as an environment variable

# Set the active project for gcloud and BigQuery CLI commands
# This ensures that subsequent gcloud and bq commands use this project.
!gcloud config set project $GOOGLE_CLOUD_PROJECT

# Print the set values for verification
print("Project:", PROJECT_ID, "| Region:", REGION)

# Done: Auth + Project/Region set

Enter your GCP Project ID: mgmt-467-471613
Updated property [core/project].
Project: mgmt-467-471613 | Region: us-central1


### Verification Prompt
Generate a short cell that prints the active project using `gcloud config get-value project` and echoes the `REGION` you set.


In [3]:
# Verify the active project
!gcloud config get-value project

# Echo the set region
import os
print("Region:", os.environ.get("REGION"))

mgmt-467-471613
Region: us-central1


**Reflection:** Why do we set `PROJECT_ID` and `REGION` at the top? What can go wrong if we don’t?

Setting the PROJECT_ID and REGION at the top ensures consistency across all the Google Cloud operations within the session. If it isn't set at the top, you can see increased costs, non-reproducible results, and operating in the wrong project.

## 1) Kaggle API — What & Why
Use Kaggle CLI for reproducible downloads. Store `kaggle.json` at `~/.kaggle/kaggle.json` with `0600` permissions to protect secrets.

### Build Prompt
Generate a **single Colab code cell** that:
- Prompts me to upload `kaggle.json`,
- Saves to `~/.kaggle/kaggle.json` with `0600` permissions,
- Prints `kaggle --version`.
Add comments about security and reproducibility.


In [5]:
# Prompt to upload the kaggle.json file
# This file contains your Kaggle API credentials and should be kept secure.
from google.colab import files
print("Upload your kaggle.json (Kaggle > Account > Create New API Token)")
uploaded = files.upload()

# Ensure the ~/.kaggle directory exists
# This is the standard location for Kaggle configuration files.
import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Save the uploaded file to the correct location
# Using the first uploaded file's name (assuming only one was uploaded).
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded[list(uploaded.keys())[0]])

# Set restrictive permissions on the API key file (owner-only read/write)
# This is a crucial security step to protect your credentials.
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Verify the Kaggle installation by printing the version
# This ensures the CLI is installed and accessible.
!kaggle --version

Upload your kaggle.json (Kaggle > Account > Create New API Token)


Saving kaggle.json to kaggle.json
Kaggle API 1.7.4.5


### Verification Prompt
Generate a one-liner that runs `kaggle --help | head -n 20` to show the CLI is ready.


**Reflection:** Why require strict `0600` permissions on API tokens? What risks are we avoiding?

## 2) Download & unzip dataset — What & Why
Keep raw files under `/content/data/raw` for predictable paths and auditing.
**Dataset:** `sayeeduddin/netflix-2025user-behavior-dataset-210k-records`

### Build Prompt
Generate a **Colab code cell** that:
- Creates `/content/data/raw`,
- Downloads the dataset to `/content/data` with Kaggle CLI,
- Unzips into `/content/data/raw` (overwrite OK),
- Lists all CSVs with sizes in a neat table.
Include comments describing each step.


In [6]:
# Create the directory for raw data
# This ensures a consistent location for the downloaded and unzipped files.
!mkdir -p /content/data/raw

# Download the dataset using the Kaggle CLI
# The dataset will be downloaded to the /content/data directory.
!kaggle datasets download -d sayeeduddin/netflix-2025user-behavior-dataset-210k-records -p /content/data

# Unzip the downloaded dataset into the raw data directory
# The -o flag allows overwriting existing files, ensuring idempotence.
!unzip -o /content/data/*.zip -d /content/data/raw

# List all CSV files in the raw data directory with their sizes
# This provides a clear inventory of the downloaded data.
!ls -lh /content/data/raw/*.csv

Dataset URL: https://www.kaggle.com/datasets/sayeeduddin/netflix-2025user-behavior-dataset-210k-records
License(s): CC0-1.0
  0% 0.00/4.02M [00:00<?, ?B/s]
100% 4.02M/4.02M [00:00<00:00, 577MB/s]
Archive:  /content/data/netflix-2025user-behavior-dataset-210k-records.zip
  inflating: /content/data/raw/README.md  
  inflating: /content/data/raw/movies.csv  
  inflating: /content/data/raw/recommendation_logs.csv  
  inflating: /content/data/raw/reviews.csv  
  inflating: /content/data/raw/search_logs.csv  
  inflating: /content/data/raw/users.csv  
  inflating: /content/data/raw/watch_history.csv  
-rw-r--r-- 1 root root 114K Aug  2 19:36 /content/data/raw/movies.csv
-rw-r--r-- 1 root root 4.5M Aug  2 19:36 /content/data/raw/recommendation_logs.csv
-rw-r--r-- 1 root root 1.8M Aug  2 19:36 /content/data/raw/reviews.csv
-rw-r--r-- 1 root root 2.2M Aug  2 19:36 /content/data/raw/search_logs.csv
-rw-r--r-- 1 root root 1.6M Aug  2 19:36 /content/data/raw/users.csv
-rw-r--r-- 1 root root 8.9M A

### Verification Prompt
Generate a snippet that asserts there are exactly **six** CSV files and prints their names.


In [7]:
import os
import glob

# Define the directory where the CSV files are located
csv_directory = '/content/data/raw/'

# Get a list of all files with the .csv extension in the directory
csv_files = glob.glob(os.path.join(csv_directory, '*.csv'))

# Assert that there are exactly six CSV files
expected_csv_count = 6
assert len(csv_files) == expected_csv_count, f"Expected {expected_csv_count} CSV files, but found {len(csv_files)}"

# Print the names of the CSV files
print(f"Found {len(csv_files)} CSV files:")
for csv_file in csv_files:
    print(os.path.basename(csv_file))

Found 6 CSV files:
movies.csv
watch_history.csv
search_logs.csv
users.csv
recommendation_logs.csv
reviews.csv


**Reflection:** Why is keeping a clean file inventory (names, sizes) useful downstream?

It helps with quick verification that you have downloaded all the expected files and their sizes are within reasonable range. It also helps with troubleshooting if files are causing issues and you need to quickly identify them.

## 3) Create GCS bucket & upload — What & Why
Stage in GCS → consistent, versionable source for BigQuery loads. Bucket names must be **globally unique**.

### Build Prompt
Generate a **Colab code cell** that:
- Creates a unique bucket in `${REGION}` (random suffix),
- Saves name to `BUCKET_NAME` env var,
- Uploads all CSVs to `gs://$BUCKET_NAME/netflix/`,
- Prints the bucket name and explains staging benefits.


In [8]:
import uuid
import os

# Generate a unique bucket name with a random suffix
# GCS bucket names must be globally unique.
bucket_name = f"mgmt467-netflix-{uuid.uuid4().hex[:8]}"
os.environ["BUCKET_NAME"] = bucket_name

# Create the GCS bucket in the specified region
# The --location flag ensures the bucket is created in the desired region.
print(f"Creating bucket: {bucket_name} in region: {os.environ['REGION']}")
!gcloud storage buckets create gs://$BUCKET_NAME --location=$REGION

# Upload all CSV files from the raw data directory to the bucket
# The destination path includes a 'netflix/' prefix within the bucket.
print(f"Uploading CSVs to gs://{bucket_name}/netflix/")
!gcloud storage cp /content/data/raw/*.csv gs://$BUCKET_NAME/netflix/

# Print the bucket name for verification
print("\nBucket created and files uploaded to:", bucket_name)

Creating bucket: mgmt467-netflix-3f2645fa in region: us-central1
Creating gs://mgmt467-netflix-3f2645fa/...
Uploading CSVs to gs://mgmt467-netflix-3f2645fa/netflix/
Copying file:///content/data/raw/movies.csv to gs://mgmt467-netflix-3f2645fa/netflix/movies.csv
Copying file:///content/data/raw/recommendation_logs.csv to gs://mgmt467-netflix-3f2645fa/netflix/recommendation_logs.csv
Copying file:///content/data/raw/reviews.csv to gs://mgmt467-netflix-3f2645fa/netflix/reviews.csv
Copying file:///content/data/raw/search_logs.csv to gs://mgmt467-netflix-3f2645fa/netflix/search_logs.csv
Copying file:///content/data/raw/users.csv to gs://mgmt467-netflix-3f2645fa/netflix/users.csv
Copying file:///content/data/raw/watch_history.csv to gs://mgmt467-netflix-3f2645fa/netflix/watch_history.csv

Average throughput: 31.6MiB/s

Bucket created and files uploaded to: mgmt467-netflix-3f2645fa


In [ ]:
# # EXAMPLE (from LLM) — GCS staging (commented)
# # import uuid, os
# # bucket_name = f"mgmt467-netflix-{uuid.uuid4().hex[:8]}"
# # os.environ["BUCKET_NAME"] = bucket_name
# # !gcloud storage buckets create gs://$BUCKET_NAME --location=$REGION
# # !gcloud storage cp /content/data/raw/* gs://$BUCKET_NAME/netflix/
# # print("Bucket:", bucket_name)
# # # Verify contents
# # !gcloud storage ls gs://$BUCKET_NAME/netflix/

### Verification Prompt
Generate a snippet that lists the `netflix/` prefix and shows object sizes.


In [9]:
import os

# List the objects in the 'netflix/' prefix of the bucket with their sizes
# The -l flag provides a detailed listing including sizes.
print(f"Listing objects in gs://{os.environ['BUCKET_NAME']}/netflix/")
!gcloud storage ls -l gs://$BUCKET_NAME/netflix/

Listing objects in gs://mgmt467-netflix-3f2645fa/netflix/
    115942  2025-10-25T20:36:14Z  gs://mgmt467-netflix-3f2645fa/netflix/movies.csv
   4695557  2025-10-25T20:36:14Z  gs://mgmt467-netflix-3f2645fa/netflix/recommendation_logs.csv
   1861942  2025-10-25T20:36:14Z  gs://mgmt467-netflix-3f2645fa/netflix/reviews.csv
   2250902  2025-10-25T20:36:14Z  gs://mgmt467-netflix-3f2645fa/netflix/search_logs.csv
   1606820  2025-10-25T20:36:14Z  gs://mgmt467-netflix-3f2645fa/netflix/users.csv
   9269425  2025-10-25T20:36:14Z  gs://mgmt467-netflix-3f2645fa/netflix/watch_history.csv
TOTAL: 6 objects, 19800588 bytes (18.88MiB)


**Reflection:** Name two benefits of staging in GCS vs loading directly from local Colab.

2 benefits of staging in GCS compared to loading directly from local Colab is the centralized store that it provide for a single accesible location for the dta. The second if the scalability and durability that is available in GCS.

## 4) BigQuery dataset & loads — What & Why
Create dataset `netflix` and load six CSVs with **autodetect** for speed (we’ll enforce schemas later).

### Build Prompt (two cells)
**Cell A:** Create (idempotently) dataset `netflix` in US multi-region; if it exists, print a friendly message.  
**Cell B:** Load tables from `gs://$BUCKET_NAME/netflix/`:
`users, movies, watch_history, recommendation_logs, search_logs, reviews`
with `--skip_leading_rows=1 --autodetect --source_format=CSV`.
Finish with row-count queries for each table.


In [10]:
# Cell A: Create (idempotently) the BigQuery dataset
DATASET = "netflix"
LOCATION = "US" # US multi-region

# Attempt to create the dataset; ignore if it already exists
# The || true part makes the command succeed even if the dataset exists, ensuring idempotency.
print(f"Attempting to create BigQuery dataset: {DATASET} in {LOCATION}")
create_dataset_command = f"bq --location={LOCATION} mk -d --description 'MGMT467 Netflix dataset' {DATASET}"
get_dataset_command = f"bq show {DATASET}"

# Execute the create command and check for success
if os.system(f"{create_dataset_command} > /dev/null 2>&1") == 0:
    print(f"Dataset '{DATASET}' created successfully.")
else:
    # If creation failed, check if it exists to provide a friendly message
    if os.system(f"{get_dataset_command} > /dev/null 2>&1") == 0:
        print(f"Dataset '{DATASET}' may already exist.")
    else:
        print(f"Failed to create dataset '{DATASET}'. Please check permissions.")

Attempting to create BigQuery dataset: netflix in US
Dataset 'netflix' may already exist.


In [9]:
# Cell B: Load tables from GCS
import os

DATASET = "netflix" # Ensure DATASET variable is set (from previous cell)

tables = {
  "users": "users.csv",
  "movies": "movies.csv",
  "watch_history": "watch_history.csv",
  "recommendation_logs": "recommendation_logs.csv",
  "search_logs": "search_logs.csv",
  "reviews": "reviews.csv",
}

bucket_name = os.environ.get("BUCKET_NAME")

if not bucket_name:
    print("Error: BUCKET_NAME environment variable is not set. Please run the GCS bucket creation cell first.")
else:
    for tbl, fname in tables.items():
      src = f"gs://{bucket_name}/netflix/{fname}"
      print(f"Loading table: {DATASET}.{tbl} from {src}")
      # Use --autodetect to infer schema and --skip_leading_rows to ignore header
      load_command = f"bq load --skip_leading_rows=1 --autodetect --source_format=CSV {DATASET}.{tbl} {src}"
      !{load_command}

    # Finish with row-count queries for each table
    print("\nRow counts after loading:")
    for tbl in tables.keys():
      print(f"Counting rows for table: {DATASET}.{tbl}")
      # Use --nouse_legacy_sql for standard SQL
      count_query = f"SELECT '{tbl}' AS table_name, COUNT(*) AS n FROM `{os.environ['GOOGLE_CLOUD_PROJECT']}.{DATASET}.{tbl}`"
      !bq query --nouse_legacy_sql "{count_query}"

Loading table: netflix.users from gs://mgmt467-netflix-1b6b1b05/netflix/users.csv
Waiting on bqjob_r435945285cb9ae0f_0000019a1b77ef1f_1 ... (1s) Current status: DONE   
Loading table: netflix.movies from gs://mgmt467-netflix-1b6b1b05/netflix/movies.csv
Waiting on bqjob_r2ae57982c949341f_0000019a1b7805dc_1 ... (1s) Current status: DONE   
Loading table: netflix.watch_history from gs://mgmt467-netflix-1b6b1b05/netflix/watch_history.csv
Waiting on bqjob_r5ce45f7822d5b16d_0000019a1b781d57_1 ... (2s) Current status: DONE   
Loading table: netflix.recommendation_logs from gs://mgmt467-netflix-1b6b1b05/netflix/recommendation_logs.csv
Waiting on bqjob_r61ac70ef7cf58a4e_0000019a1b783744_1 ... (2s) Current status: DONE   
Loading table: netflix.search_logs from gs://mgmt467-netflix-1b6b1b05/netflix/search_logs.csv
Waiting on bqjob_r2de2477503b36c01_0000019a1b7851dd_1 ... (2s) Current status: DONE   
Loading table: netflix.reviews from gs://mgmt467-netflix-1b6b1b05/netflix/reviews.csv
Waiting on 

### Verification Prompt
Generate a single query that returns `table_name, row_count` for all six tables in `${GOOGLE_CLOUD_PROJECT}.netflix`.


In [11]:
%%bigquery
SELECT 'users' as table_name, COUNT(*) as row_count FROM `mgmt-467-471613.netflix.users`
UNION ALL
SELECT 'movies' as table_name, COUNT(*) as row_count FROM `mgmt-467-471613.netflix.movies`
UNION ALL
SELECT 'watch_history' as table_name, COUNT(*) as row_count FROM `mgmt-467-471613.netflix.watch_history`
UNION ALL
SELECT 'recommendation_logs' as table_name, COUNT(*) as row_count FROM `mgmt-467-471613.netflix.recommendation_logs`
UNION ALL
SELECT 'search_logs' as table_name, COUNT(*) as row_count FROM `mgmt-467-471613.netflix.search_logs`
UNION ALL
SELECT 'reviews' as table_name, COUNT(*) as row_count FROM `mgmt-467-471613.netflix.reviews`

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count
0,recommendation_logs,156000
1,movies,3120
2,users,30900
3,reviews,46350
4,watch_history,315000
5,search_logs,79500


**Reflection:** When is `autodetect` acceptable? When should you enforce explicit schemas and why?

## 5) Data Quality (DQ) — Concepts we care about
- **Missingness** (MCAR/MAR/MNAR). Impute vs drop. Add `is_missing_*` indicators.
- **Duplicates** (exact vs near). Double-counted engagement corrupts labels & KPIs.
- **Outliers** (IQR). Winsorize/cap vs robust models. Always **flag** and explain.
- **Reproducibility**. Prefer `CREATE OR REPLACE` and deterministic keys.


### 5.1 Missingness (users) — What & Why
Measure % missing and check if missingness depends on another variable (MAR) → potential bias & instability.

### Build Prompt
Generate **two BigQuery SQL cells**:
1) Total rows and % missing in `region`, `plan_tier`, `age_band` from `users`.
2) `% plan_tier missing by region` ordered descending. Add comments on MAR.


In [1]:
%%bigquery
-- Total rows and % missing in country, subscription_plan, age from users
WITH base AS (
  SELECT
    COUNT(*) AS total_rows,
    COUNTIF(country IS NULL) AS missing_region,
    COUNTIF(subscription_plan IS NULL) AS missing_plan_tier,
    COUNTIF(age IS NULL) AS missing_age_band
  FROM
    `mgmt-467-471613.netflix.users`
)
SELECT
  total_rows,
  ROUND(100 * missing_region / total_rows, 2) AS pct_missing_region,
  ROUND(100 * missing_plan_tier / total_rows, 2) AS pct_missing_plan_tier,
  ROUND(100 * missing_age_band / total_rows, 2) AS pct_missing_age_band
FROM base


ERROR:
 404 POST https://bigquery.googleapis.com/bigquery/v2/projects//jobs?prettyPrint=false: Request couldn't be served.

Location: None
Job ID: 6d8fde90-ce92-4186-b580-6d4102b6bd7b



In [12]:
%%bigquery
-- % plan_tier missing by country ordered descending (to check for MAR)
-- Missingness in plan_tier might depend on the country (MAR - Missing At Random)
SELECT
  country,
  COUNT(*) AS total_in_region,
  COUNTIF(subscription_plan IS NULL) AS missing_plan_tier_in_region,
  ROUND(100 * COUNTIF(subscription_plan IS NULL) / COUNT(*), 2) AS pct_missing_plan_tier
FROM
  `mgmt-467-471613.netflix.users`
GROUP BY
  country
ORDER BY
  pct_missing_plan_tier DESC

Query is running:   0%|          |

Downloading:   0%|          |

,country,total_in_region,missing_plan_tier_in_region,pct_missing_plan_tier
0,Canada,9288,0,0.0
1,USA,21612,0,0.0


### Verification Prompt
Generate a query that prints the three missingness percentages from (1), rounded to two decimals.


**Reflection:** Which columns are most missing? Hypothesize MCAR/MAR/MNAR and why.

### 5.2 Duplicates (watch_history) — What & Why
Find exact duplicate interaction records and keep **one best** per group (deterministic policy).

### Build Prompt
Generate **two BigQuery SQL cells**:
1) Report duplicate groups on `(user_id, movie_id, event_ts, device_type)` with counts (top 20).
2) Create table `watch_history_dedup` that keeps one row per group (prefer higher `progress_ratio`, then `minutes_watched`). Add comments.


In [16]:
%%bigquery
-- Report duplicate groups on (user_id, movie_id, event_ts, device_type) with counts (top 20)
SELECT user_id, movie_id, action, device_type, COUNT(*) AS dup_count
FROM `mgmt-467-471613.netflix.watch_history`
GROUP BY user_id, movie_id, action, device_type
HAVING dup_count > 1
ORDER BY dup_count DESC
LIMIT 20;

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,movie_id,action,device_type,dup_count
0,user_03310,movie_0640,stopped,Smart TV,12
1,user_00391,movie_0893,stopped,Laptop,12
2,user_00472,movie_0719,started,Laptop,9
3,user_02976,movie_0987,paused,Desktop,9
4,user_01469,movie_0237,stopped,Laptop,9
5,user_02028,movie_0037,paused,Desktop,9
6,user_07594,movie_0133,paused,Laptop,9
7,user_02359,movie_0108,completed,Desktop,9
8,user_03348,movie_0688,paused,Desktop,9
9,user_03898,movie_0500,stopped,Desktop,9


In [18]:
%%bigquery
-- Create table watch_history_dedup that keeps one row per group
-- Keeps one row per group (user_id, movie_id, event_ts, device_type)
-- Preference given to higher progress_ratio, then minutes_watched for tie-breaking
CREATE OR REPLACE TABLE `mgmt-467-471613.netflix.watch_history_dedup` AS
SELECT * EXCEPT(rk) FROM (
  SELECT h.*,
         ROW_NUMBER() OVER (
           PARTITION BY user_id, movie_id, action, device_type
           ORDER BY progress_percentage DESC, watch_duration_minutes DESC
         ) AS rk
  FROM `mgmt-467-471613.netflix.watch_history` h
)
WHERE rk = 1;

Query is running:   0%|          |

""


### Verification Prompt
Generate a before/after count query comparing raw vs `watch_history_dedup`.


In [19]:
%%bigquery
-- Compare row counts of raw and deduped watch history tables
SELECT
  'watch_history' AS table_name,
  COUNT(*) AS row_count
FROM
  `mgmt-467-471613.netflix.watch_history`
UNION ALL
SELECT
  'watch_history_dedup' AS table_name,
  COUNT(*) AS row_count
FROM
  `mgmt-467-471613.netflix.watch_history_dedup`

Query is running:   0%|          |

Downloading:   0%|          |

,table_name,row_count
0,watch_history_dedup,99958
1,watch_history,315000


**Reflection:** Why do duplicates arise (natural vs system-generated)? How do they corrupt labels and KPIs?

Duplicates in data can arise from various sources. **Natural duplicates** might occur when a real-world event is recorded multiple times, perhaps through different channels or at slightly different times (e.g., a user accidentally clicking "play" twice). **System-generated duplicates** often result from issues in data pipelines, ETL processes, or database operations.

Duplicates can significantly **corrupt labels and KPIs** because they inflate counts and distort aggregations.

### 5.3 Outliers (minutes_watched) — What & Why
Estimate extreme values via IQR; report % outliers; **winsorize** to P01/P99 for robustness while also **flagging** extremes.

### Build Prompt
Generate **two BigQuery SQL cells**:
1) Compute IQR bounds for `minutes_watched` on `watch_history_dedup` and report % outliers.
2) Create `watch_history_robust` with `minutes_watched_capped` capped at P01/P99; return quantile summaries before/after.


In [20]:
%%bigquery
-- Compute IQR bounds for minutes_watched on watch_history_dedup and report % outliers.
WITH dist AS (
  SELECT
    APPROX_QUANTILES(watch_duration_minutes, 4)[OFFSET(1)] AS q1,
    APPROX_QUANTILES(watch_duration_minutes, 4)[OFFSET(3)] AS q3
  FROM
    `mgmt-467-471613.netflix.watch_history_dedup`
),
bounds AS (
  SELECT
    q1,
    q3,
    (q3 - q1) AS iqr,
    q1 - 1.5 * (q3 - q1) AS lo,
    q3 + 1.5 * (q3 - q1) AS hi
  FROM dist
)
SELECT
  COUNTIF(h.watch_duration_minutes < b.lo OR h.watch_duration_minutes > b.hi) AS outliers,
  COUNT(*) AS total,
  ROUND(
    100 * COUNTIF(
      h.watch_duration_minutes < b.lo
      OR h.watch_duration_minutes > b.hi
    ) / COUNT(*),
    2
  ) AS pct_outliers
FROM
  `mgmt-467-471613.netflix.watch_history_dedup` AS h
CROSS JOIN
  bounds AS b

Query is running:   0%|          |

Downloading:   0%|          |

,outliers,total,pct_outliers
0,3499,99958,3.5


In [24]:
%%bigquery
-- Create table watch_history_robust with minutes_watched_capped at P01/P99; return quantile summaries before/after.
CREATE OR REPLACE TABLE `mgmt-467-471613.netflix.watch_history_robust` AS
WITH q AS (
  SELECT
    APPROX_QUANTILES(watch_duration_minutes, 100)[OFFSET(1)] AS p01,
    APPROX_QUANTILES(watch_duration_minutes, 100)[OFFSET(99)] AS p99
  FROM
    `mgmt-467-471613.netflix.watch_history_dedup`
)
SELECT
  h.*,
  GREATEST(q.p01, LEAST(q.p99, h.watch_duration_minutes)) AS minutes_watched_capped
FROM
  `mgmt-467-471613.netflix.watch_history_dedup` AS h,
  q;

-- Quantiles before vs after
WITH before AS (
  SELECT
    'before' AS which,
    APPROX_QUANTILES(watch_duration_minutes, 5) AS q
  FROM
    `mgmt-467-471613.netflix.watch_history_dedup`
),
after AS (
  SELECT
    'after' AS which,
    APPROX_QUANTILES(minutes_watched_capped, 5) AS q
  FROM
    `mgmt-467-471613.netflix.watch_history_robust`
)
SELECT
  which,
  q[OFFSET(0)] AS min_value,
  q[OFFSET(2)] AS median_value,
  q[OFFSET(4)] AS max_value
FROM before
UNION ALL
SELECT
  which,
  q[OFFSET(0)] AS min_value,
  q[OFFSET(2)] AS median_value,
  q[OFFSET(4)] AS max_value
FROM after;

Query is running:   0%|          |

Downloading:   0%|          |

,which,min_value,median_value,max_value
0,before,0.2,41.7,91.7
1,after,4.4,41.6,92.2


### Verification Prompt
Generate a query that shows min/median/max before vs after capping.


In [25]:
%%bigquery
-- Quantiles before vs after capping
WITH before AS (
  SELECT
    'before' AS which,
    APPROX_QUANTILES(watch_duration_minutes, 5) AS q
  FROM
    `mgmt-467-471613.netflix.watch_history_dedup`
),
after AS (
  SELECT
    'after' AS which,
    APPROX_QUANTILES(minutes_watched_capped, 5) AS q
  FROM
    `mgmt-467-471613.netflix.watch_history_robust`
)
SELECT
  which,
  q[OFFSET(0)] AS min_value,
  q[OFFSET(2)] AS median_value,
  q[OFFSET(4)] AS max_value
FROM before
UNION ALL
SELECT
  which,
  q[OFFSET(0)] AS min_value,
  q[OFFSET(2)] AS median_value,
  q[OFFSET(4)] AS max_value
FROM after;

Query is running:   0%|          |

Downloading:   0%|          |

,which,min_value,median_value,max_value
0,before,0.2,41.7,91.7
1,after,4.4,41.6,92.2


**Reflection:** When might capping be harmful? Name a model type less sensitive to outliers and why.

When might capping be harmful?

Capping, while useful for reducing the influence of extreme values, can be harmful when:

* **Outliers are meaningful:** If the extreme values represent genuine, important
data points (e.g., a user who watched for a very long duration because they binged an entire series), capping them can lead to a loss of valuable information and distort the true distribution of the data. This can hide important patterns or insights.
* **The distribution is naturally skewed:** For data with a naturally skewed distribution, like income or website visits, applying aggressive capping based on percentiles can flatten the distribution and misrepresent the data's characteristics.
* **Interpretability is key:** Capped values lose their original meaning. It's harder to interpret a capped value of, say, 99 minutes, than the original value of 500 minutes if that 500 minutes represents a real user behavior you want to understand.

### 5.4 Business anomaly flags — What & Why
Human-readable flags help both product decisioning and ML features (e.g., binge behavior).

### Build Prompt
Generate **three BigQuery SQL cells** (adjust if columns differ):
1) In `watch_history_robust`, compute and summarize `flag_binge` for sessions > 8 hours.
2) In `users`, compute and summarize `flag_age_extreme` if age can be parsed from `age_band` (<10 or >100).
3) In `movies`, compute and summarize `flag_duration_anomaly` where `duration_min` < 15 or > 480 (if exists).
Each cell should output count and percentage and include 1–2 comments.


In [35]:
%%bigquery
-- Compute and summarize flag_binge for sessions > 8 hours in watch_history_robust
SELECT
  COUNTIF(minutes_watched_capped > 8*60) AS sessions_over_8h,
  COUNT(*) AS total,
  ROUND(100*COUNTIF(minutes_watched_capped > 8*60)/COUNT(*),2) AS pct
FROM `mgmt-467-471613.netflix.watch_history_robust`;

Query is running:   0%|          |

Downloading:   0%|          |

,sessions_over_8h,total,pct
0,0,99958,0.0


In [36]:
%%bigquery
-- Compute and summarize flag_age_extreme if age can be parsed from age_band (<10 or >100) in users
SELECT
  COUNTIF(SAFE_CAST(REGEXP_EXTRACT(CAST(age AS STRING), r'\d+') AS INT64) < 10 OR
          SAFE_CAST(REGEXP_EXTRACT(CAST(age AS STRING), r'\d+') AS INT64) > 100) AS extreme_age_rows,
  COUNT(*) AS total,
  ROUND(100*COUNTIF(SAFE_CAST(REGEXP_EXTRACT(CAST(age AS STRING), r'\d+') AS INT64) < 10 OR
                    SAFE_CAST(REGEXP_EXTRACT(CAST(age AS STRING), r'\d+') AS INT64) > 100)/COUNT(*),2) AS pct
FROM `mgmt-467-471613.netflix.users`;

Query is running:   0%|          |

Downloading:   0%|          |

,extreme_age_rows,total,pct
0,537,30900,1.74


In [37]:
%%bigquery
-- Compute and summarize flag_duration_anomaly where duration_min < 15 or > 480 in movies
-- Assuming 'duration_min' column exists in the movies table.
SELECT
  COUNTIF(duration_minutes < 15) AS titles_under_15m,
  COUNTIF(duration_minutes > 480) AS titles_over_8h, -- 480 minutes = 8 hours
  COUNT(*) AS total,
  ROUND(100*COUNTIF(duration_minutes < 15 OR duration_minutes > 480)/COUNT(*),2) AS pct_anomaly
FROM `mgmt-467-471613.netflix.movies`;

Query is running:   0%|          |

Downloading:   0%|          |

,titles_under_15m,titles_over_8h,total,pct_anomaly
0,36,33,3120,2.21


### Verification Prompt
Generate a single compact summary query that returns two columns per flag: `flag_name, pct_of_rows`.


**Reflection:** Which anomaly flag is most common? Which would you keep as a feature and why?

## 6) Save & submit — What & Why
Reproducibility: save artifacts and document decisions so others can rerun and audit.

Based on the outputs of the anomaly flag queries:

* **flag_binge** (sessions > 8 hours): 0.0% of sessions.
* **flag_age_extreme** (age < 10 or > 100): 1.74% of users.
* **flag_duration_anomaly** (duration < 15 or > 480 minutes): 2.21% of movies.

The **flag_duration_anomaly** (movies with unusually short or long durations) is the most common anomaly flag, affecting 2.21% of the movie titles.

Which flag I would keep as a feature depends on the specific business problem or ML task I'm trying to solve. However, the **flag_age_extreme** could be particularly valuable as a feature, even though it's less common than the duration anomaly.

### Build Prompt
Generate a checklist (Markdown) students can paste at the end:
- Save this notebook to the team Drive.
- Export a `.sql` file with your DQ queries and save to repo.
- Push notebook + SQL to the **team GitHub** with a descriptive commit.
- Add a README with your `PROJECT_ID`, `REGION`, bucket, dataset, and today’s row counts.


## Grading rubric (quick)
- Profiling completeness (30)  
- Cleaning policy correctness & reproducibility (40)  
- Reflection/insight (20)  
- Hygiene (naming, verification, idempotence) (10)
